# BC카드 공모전 — 5단계 외부데이터 수집 (구글트렌드)

이 노트북은 H1(선행성) 가설 검증에 쓸 외부데이터(구글트렌드 검색관심도)를 후보지별로 수집한다.

**대상**: 3단계에서 확정한 8개 후보지 — 플래그십(청송군·봉화군·고성군(강원)·영월군), 유지후보(영양군·의성군·양양군), 보조사례(옹진군)

**키워드 이원화(9/15 결정)**: 관광 트랙은 `지역명+여행`/`지역명+맛집`, 이주노동 트랙은 `지역명+계절근로자` — 카드신호와 어느 쪽이 더 강하게 상관되는지 비교해서 관광/이주노동 신호를 사후적으로 구분하는 데 쓴다.

**주의**: 강원 고성군은 경남 고성군과 동명이라, 검색어에 지역명만 넣으면 두 지역이 섞일 수 있음 — 결과 해석 시 유의(가능하면 "고성군 강원" 같은 보조 키워드로 교차 확인 권장).

실행 후 `google_trends_monthly.csv`를 드라이브 `BC카드` 폴더에 저장한다 — 별도로 파일을 안 보내줘도 Claude가 드라이브에서 바로 읽어감.

## 0. 환경 설정

In [ ]:
!pip install pytrends -q

import pandas as pd
import time
from pytrends.request import TrendReq

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/BC카드'

pytrends = TrendReq(hl='ko-KR', tz=540, timeout=(10, 25), retries=2, backoff_factor=0.5)


## 1. 후보지 목록 & 키워드 정의

- Track A(관광): 8개 후보지 전체 × [여행, 맛집]
- Track B(이주노동): 계절근로자 정황이 의심/확인된 6개 후보지 × [계절근로자]

In [ ]:
CANDIDATES = ['청송군', '봉화군', '고성군', '영월군', '영양군', '의성군', '양양군', '옹진군']
TRACK_A_SUFFIXES = ['여행', '맛집']
TRACK_B_REGIONS = ['청송군', '봉화군', '고성군', '영월군', '영양군', '의성군']  # 계절근로자 정황 있는 지역만
TRACK_B_SUFFIX = '계절근로자'

TIMEFRAME = '2026-01-01 2026-06-30'
GEO = 'KR'


## 2. 수집 함수

구글트렌드는 429(rate limit)가 잦아서 요청 사이에 딜레이를 두고, 실패 시 한 번 재시도한다. 6개월 미만 구간은 일별 데이터가 나오므로 월별 평균으로 리샘플링한다.

In [ ]:
def fetch_monthly_trend(keyword, timeframe=TIMEFRAME, geo=GEO, max_retries=3, sleep_sec=8):
    for attempt in range(max_retries):
        try:
            pytrends.build_payload([keyword], timeframe=timeframe, geo=geo)
            df = pytrends.interest_over_time()
            if df.empty:
                print(f'  [{keyword}] 결과 없음 (검색량 부족 가능)')
                return None
            if 'isPartial' in df.columns:
                df = df.drop(columns=['isPartial'])
            df.index = pd.to_datetime(df.index)
            monthly = df.resample('MS').mean()
            return monthly[keyword]
        except Exception as e:
            print(f'  [{keyword}] 시도 {attempt+1}/{max_retries} 실패: {e}')
            time.sleep(sleep_sec * (attempt + 1))
    print(f'  [{keyword}] 최종 실패 — 건너뜀')
    return None


## 3. Track A (관광) 수집

In [ ]:
results = {}

for region in CANDIDATES:
    for suf in TRACK_A_SUFFIXES:
        kw = f'{region} {suf}'
        print('수집 중:', kw)
        s = fetch_monthly_trend(kw)
        if s is not None:
            results[kw] = s
        time.sleep(5)  # 요청 간 간격 — 너무 짧으면 429 잘 남


## 4. Track B (이주노동) 수집

In [ ]:
for region in TRACK_B_REGIONS:
    kw = f'{region} {TRACK_B_SUFFIX}'
    print('수집 중:', kw)
    s = fetch_monthly_trend(kw)
    if s is not None:
        results[kw] = s
    time.sleep(5)


## 5. 결과 정리 & 저장

In [ ]:
trends_df = pd.DataFrame(results)
trends_df.index = trends_df.index.strftime('%Y%m')
trends_df.index.name = 'STRD_YYMM'

print('수집된 키워드 수:', len(trends_df.columns))
print('수집 실패(누락)한 키워드:', [kw for kw in list(results.keys())])
trends_df


In [ ]:
out_path = f'{BASE_DIR}/google_trends_monthly.csv'
trends_df.to_csv(out_path, encoding='utf-8-sig')
print('저장 완료:', out_path)


## 참고 — 실패/재실행

- 구글트렌드는 짧은 시간에 요청이 몰리면 429가 자주 뜬다. 전체가 실패하면 런타임을 재시작하고 시간을 좀 두고 다시 실행할 것.
- 결과가 비어있는(None) 키워드는 검색량 자체가 너무 적어서 구글이 데이터를 안 주는 경우 — 그 후보지는 이 키워드로는 검증 불가 처리.
- 저장된 CSV는 `BC카드` 드라이브 폴더에 있으므로 별도로 파일을 전달할 필요 없음.